<div style="text-align:center; margin-top:25px; margin-bottom:25px;">

<h1 style="
color:#2C3E50;
font-family:Georgia;
font-size:38px;
margin-bottom:8px;">
Appendix A — XAI ATEFEH System
</h1>

<h3 style="
color:#7F8C8D;
font-family:Georgia;
font-weight:normal;
font-size:22px;
margin-top:0;">
Jupyter Notebook Implementation
</h3>

<hr style="
width:65%;
margin-top:20px;
border:1px solid #D5D8DC;">

<p style="
font-size:16px;
color:#566573;
margin-top:15px;">
ATEFEH Pipeline · Full Annotated Source

</p>

</div>

<h2 style="color:#2C3E50;">Introduction</h2>
<p style="font-size:16px; line-height:1.9; text-align:justify;">
The ATEFEH framework is designed as a collaborative multi-agent
Explainable Artificial Intelligence (XAI) system for automated
machine learning notebook analysis.
Three intelligent agents work together sequentially throughout
the pipeline:
</p>
<ul style="font-size:16px; line-height:1.9;">
<li>
<b>User-Agent</b> receives the notebook path from the user
and delivers the final generated explanation back to the user.
</li>
<li>
<b>ATEFEH-Agent</b> validates the notebook structure,
reads each cell's source code and any pre-existing outputs
already stored in the notebook file, and constructs a
structured XAI summary — without executing the notebook or
requiring access to its original dataset.
</li>
<li>
<b>LLM-Agent</b> sends the generated summary to the Ollama
large language model and returns a quality-checked
natural-language explanation.
</li>
</ul>
<p style="font-size:16px; line-height:1.9; text-align:justify;">
The overall objective of the system is to automate the
interpretation of machine learning workflows and generate
human-readable explanations directly from their source code
and structure, independent of external datasets.
</p>

<h2 style="color:#2C3E50;">
Cell 1 — Install Dependencies
</h2>

<p style="font-size:16px; line-height:1.9; text-align:justify;">

This cell installs all required Python libraries before executing
the remainder of the notebook. The dependencies support notebook
processing, automated execution, and communication with the
Ollama local large language model API.

</p>

<ul style="font-size:16px; line-height:1.9;">

<li>
<b>nbformat</b> — reads and writes Jupyter <code>.ipynb</code> notebook files.
</li>

<li>
<b>nbconvert</b> — executes notebooks programmatically and captures outputs.
</li>

<li>
<b>requests</b> — communicates with the Ollama REST API running on port 11434.
</li>

</ul>

<div style="
background-color:#F8F9F9;
padding:14px;
border-left:5px solid #5DADE2;
border-radius:8px;
margin-top:15px;">

<b>Important:</b>
Ollama must be installed and running locally before executing
the pipeline.

</div>

In [26]:
!pip install nbformat nbconvert requests --quiet


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


<h2 style="color:#2C3E50;">
Cell 2 — Import Libraries
</h2>

<p style="font-size:16px; line-height:1.9; text-align:justify;">

This cell imports the standard Python libraries and third-party
dependencies required throughout the ATEFEH pipeline.
These libraries support file handling, notebook processing,
logging, warning management, and communication with the
Ollama large language model API.

</p>

<p style="font-size:16px; line-height:1.9; text-align:justify;">

Warnings are suppressed to maintain clean and readable notebook
outputs during execution. In addition, the logging system is
initialised in this section. The logging format includes the
execution timestamp, log level, and message content, and is
automatically inherited by all subsequent cells in the pipeline.

</p>

<div style="
background-color:#F8F9F9;
padding:14px;
border-left:5px solid #5DADE2;
border-radius:8px;
margin-top:15px;">

<b>Note:</b>
The logging configuration defined in this cell is shared globally
across the entire notebook execution process.

</div>

In [27]:
import os          # file existence checks and path operations
import json        # serialize/deserialize result dicts to disk
import logging     # structured console output with timestamps
import warnings    # suppress noisy deprecation warnings from libraries
from pathlib import Path  # object-oriented filesystem paths
 
import requests                                         # HTTP calls to Ollama REST API
import nbformat                                         # parse and write .ipynb notebook files
from nbconvert.preprocessors import ExecutePreprocessor # run all cells in a live Python kernel
import sys
import geopandas as gpd
from shapely.geometry import Point, Polygon


# Suppress all warnings so pipeline output stays readable
warnings.filterwarnings("ignore")
 
# Configure logging once — every logger in every cell inherits this format
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)
 
logger.info("All imports successful")

2026-07-19 18:29:16,929 - INFO - All imports successful


<h2 style="color:#2C3E50;">
Cell 3 — Configuration
</h2>

<p style="font-size:16px; line-height:1.9; text-align:justify;">

This cell defines the core configuration parameters used
throughout the ATEFEH pipeline. These settings specify the
target notebook, the Ollama API endpoint, the selected
large language model, and the directory used for storing
generated outputs and execution artefacts.

</p>

<ul style="font-size:16px; line-height:1.9;">

<li>
<b>NOTEBOOK_PATH</b> — specifies the path to the
<code>.ipynb</code> notebook file that will be analysed
and explained by the ATEFEH system.
</li>

<li>
<b>OLLAMA_URL</b> — defines the default REST API endpoint
used to communicate with the locally running Ollama server.
No modification is required for a standard installation.
</li>

<li>
<b>OLLAMA_MODEL</b> — specifies the large language model
used for explanation generation. The default model is
<code>llama3</code>, while alternatives such as
<code>mistral</code> and <code>phi3</code> may also be used.
</li>

<li>
<b>OUTPUT_DIR</b> — defines the directory where all
generated pipeline artefacts, including executed notebooks,
JSON outputs, and generated explanations, are stored.
</li>

</ul>

<div style="
background-color:#F8F9F9;
padding:14px;
border-left:5px solid #5DADE2;
border-radius:8px;
margin-top:15px;">

<b>Important:</b>
All configuration values defined in this section are shared
globally across the entire ATEFEH pipeline.

</div>

In [51]:
NOTEBOOK_PATH = "Dataset.ipynb"   # <-- change this to your actual file path
 
OLLAMA_URL   = "http://localhost:11434/api/generate"  # standard Ollama endpoint
OLLAMA_MODEL = "llama3"                               # LLM model to use locally
 
OUTPUT_DIR = Path("notebook_outputs")   # all output files go here
OUTPUT_DIR.mkdir(exist_ok=True)         # create the folder silently if it doesn't exist yet
 
logger.info(f"Notebook path : {NOTEBOOK_PATH}")
logger.info(f"LLM model     : {OLLAMA_MODEL}")
logger.info(f"Output dir    : {OUTPUT_DIR}")
 

2026-07-19 20:18:36,519 - INFO - Notebook path : Dataset.ipynb
2026-07-19 20:18:36,524 - INFO - LLM model     : llama3
2026-07-19 20:18:36,528 - INFO - Output dir    : notebook_outputs


<h2 style="color:#2C3E50;">
User-Agent
</h2>

<p style="font-size:16px; line-height:1.9; text-align:justify;">

The User-Agent is responsible for interacting directly with the end user and acts as the entry and exit point of the ATEFEH pipeline.

It performs two main roles:
</p>

<ul style="font-size:16px; line-height:1.9;">

<li>
<b>Step 1 — Submission:</b> receives the path of the target Jupyter notebook and submits it to the pipeline for processing.
</li>

<li>
<b>Step 6 — Delivery:</b> receives the final generated explanation from the system and returns it to the user in a readable format.
</li>

</ul>

<hr style="margin-top:15px; margin-bottom:15px;">

<h3 style="color:#34495E;">
Cell 4 — submit_ml_program (User-Agent · Step 1)
</h3>

<p style="font-size:16px; line-height:1.9; text-align:justify;">

This function represents the first step of the pipeline. It accepts the file-system path of the target <code>.ipynb</code> notebook, verifies its existence, and constructs a structured submission dictionary containing metadata such as file path, notebook title, and file size in kilobytes.

</p>

<ul style="font-size:16px; line-height:1.9;">

<li><b>Input:</b> <code>ipynb_path (str)</code> — path to the notebook file</li>
<li><b>Output:</b> <code>submission (dict)</code> — structured metadata including path, title, and size</li>
</ul>

<div style="
background-color:#F8F9F9;
padding:14px;
border-left:5px solid #5DADE2;
border-radius:8px;
margin-top:15px;">

<b>Note:</b>
This step does not execute the notebook; it only registers and prepares it for processing in the ATEFEH pipeline.

</div>

In [29]:
def submit_ml_program(ipynb_path: str) -> dict:
    """
    Receive the ML notebook path from the user and build a submission dict.
 
    Parameters
    ----------
    ipynb_path : str
        File-system path to the target .ipynb notebook.
        Example: "models/heart_disease_predictions.ipynb"
 
    Returns
    -------
    dict
        "path"    (str)   — original ipynb_path as passed in by the caller
        "title"   (str)   — filename stem without extension
                            e.g. "heart_disease_predictions"
        "size_kb" (float) — file size in KB rounded to 1 decimal
                            e.g. 142.7 for a 146,124-byte file
 
    Raises
    ------
    FileNotFoundError
        If ipynb_path does not point to an existing file.
    """
    logger.info(f"[User-Agent] Submitting notebook: {ipynb_path}")
 
    # --- Guard: fail fast before any processing if the file is missing ---
    if not os.path.exists(ipynb_path):
        logger.error(f"File not found: {ipynb_path}")
        raise FileNotFoundError(f"File not found: {ipynb_path}")
 
    # --- Build the submission metadata dict ---
    submission = {
        "path"   : ipynb_path,
        "title"  : Path(ipynb_path).stem,                        # e.g. "my_model" from "my_model.ipynb"
        "size_kb": round(os.path.getsize(ipynb_path) / 1024, 1)  # convert bytes -> KB
    }
 
    logger.info(f"[User-Agent] Submission ready | title={submission['title']} | size={submission['size_kb']} KB")
    return submission
 
logger.info("submit_ml_program ready")

2026-07-19 18:29:20,327 - INFO - submit_ml_program ready


<h3 style="color:#2C3E50;">
Cell 5 — receive_explanation (User-Agent · Step 6)
</h3>

<p style="font-size:16px; line-height:1.9; text-align:justify;">

This function represents the final step of the User-Agent within the ATEFEH pipeline. It is responsible for receiving the final natural-language explanation generated by the LLM-Agent and delivering it to the end user in a structured and persistent format.

In addition to displaying the explanation in the console logs, the function also stores the output as a text file for later review and documentation purposes.

</p>

<ul style="font-size:16px; line-height:1.9;">

<li>
<b>Input:</b> <code>explanation (str)</code> — full natural-language explanation generated by the LLM-Agent
</li>

<li>
<b>Input:</b> <code>title (str)</code> — notebook stem used to construct the output filename
</li>

<li>
<b>Output:</b> None — this function performs side effects only (logging + file writing)
</li>

</ul>

<div style="
background-color:#F8F9F9;
padding:14px;
border-left:5px solid #5DADE2;
border-radius:8px;
margin-top:15px;">

<b>Note:</b>
The explanation is saved as a <code>.txt</code> file inside the configured output directory to ensure reproducibility and traceability of results.

</div>

In [30]:
def receive_explanation(explanation: str, title: str) -> None:
    """
    Deliver the finished explanation to the end user (User-Agent · Step 6).
 
    Parameters
    ----------
    explanation : str
        The complete XAI explanation from generate_explanation.
        May contain section headers: TASK, RESULTS, BEHAVIOUR, LIMITATIONS.
    title : str
        Notebook stem. Used to build: OUTPUT_DIR/<title>_explanation.txt
 
    Returns
    -------
    None
        Side effects:
          - Console: each line emitted as INFO log entry
          - Disk: written to OUTPUT_DIR/<title>_explanation.txt as UTF-8
    """
    logger.info("[User-Agent] Explanation received from ATEFEH")
    logger.info("-" * 60)
 
    # splitlines() handles \n, \r\n, and \r line endings uniformly
    for line in explanation.splitlines():
        logger.info(line)
 
    logger.info("-" * 60)
 
    out_file = OUTPUT_DIR / f"{title}_explanation.txt"
    with open(out_file, "w", encoding="utf-8") as f:
        f.write(explanation)
 
    logger.info(f"[User-Agent] Explanation saved: {out_file}")
 
logger.info("receive_explanation ready")
 

2026-07-19 18:29:21,610 - INFO - receive_explanation ready


<h2 style="color:#2C3E50;">
ATEFEH-Agent
</h2>

<p style="font-size:16px; line-height:1.9; text-align:justify;">

The ATEFEH-Agent is responsible for all core processing tasks within the pipeline. It acts as the computational engine that validates, executes, and transforms the submitted Jupyter notebook into structured information suitable for explainable AI analysis.

</p>

<ul style="font-size:16px; line-height:1.9;">

<li>
<b>Step 2 — Validation:</b> verifies the submitted notebook before any execution or processing begins.
</li>

<li>
<b>Step 4a — Execution:</b> runs all notebook cells and captures their outputs.
</li>

<li>
<b>Step 4b — Extraction:</b> extracts and structures raw outputs from executed cells.
</li>

<li>
<b>Step 4c — Summarization:</b> saves extracted outputs and builds a structured XAI summary for the LLM-Agent.
</li>

</ul>

<hr style="margin-top:15px; margin-bottom:15px;">

<h3 style="color:#34495E;">
Cell 6 — validate_input (ATEFEH-Agent · Step 2)
</h3>

<p style="font-size:16px; line-height:1.9; text-align:justify;">

This function validates the submitted notebook before any further processing takes place. It ensures that the input is structurally correct, safely readable, and contains at least one executable code cell.

</p>

<ul style="font-size:16px; line-height:1.9;">

<li><b>Input:</b> <code>submission (dict)</code> — dictionary produced by <code>submit_ml_program</code></li>
<li><b>Output:</b> <code>(is_valid, message, nb)</code> — validation status, message, and parsed notebook object</li>
</ul>

<div style="
background-color:#F8F9F9;
padding:14px;
border-left:5px solid #5DADE2;
border-radius:8px;
margin-top:15px;">

<b>Note:</b>
If validation fails, the pipeline is terminated immediately to prevent unsafe or invalid execution.

</div>

In [31]:
def validate_input(submission: dict) -> tuple:
    """
    Validate the submitted notebook (ATEFEH-Agent · Step 2).
 
    Parameters
    ----------
    submission : dict
        Must contain "path" (str). Other keys are ignored.
 
    Returns
    -------
    tuple of (bool, str, NotebookNode or None)
        Element 0 — is_valid : bool
            True if all four checks passed; False otherwise.
        Element 1 — message : str
            "Valid"
            "File not found"
            "File must be .ipynb"
            "Invalid notebook format: <e>"
            "Notebook contains no code cells"
        Element 2 — nb : NotebookNode or None
            Parsed notebook object (nbformat v4) on success.
            None on any failure.
    """
    logger.info("[ATEFEH] Starting input validation")
 
    path = submission.get("path", "")
 
    # Check 1: file must exist on disk
    if not os.path.exists(path):
        logger.error(f"Validation failed: file not found -- {path}")
        return False, "File not found", None
 
    # Check 2: must be a Jupyter Notebook file
    if not path.endswith(".ipynb"):
        logger.error("Validation failed: file must be .ipynb")
        return False, "File must be .ipynb", None
 
    # Check 3: file must be parseable as a valid notebook
    try:
        with open(path, "r", encoding="utf-8") as f:
            nb = nbformat.read(f, as_version=4)
    except Exception as e:
        logger.error(f"Validation failed: invalid format -- {e}")
        return False, f"Invalid notebook format: {e}", None
 
    # Check 4: notebook must contain at least one executable code cell
    code_cells = [c for c in nb.cells if c.cell_type == "code"]
    if len(code_cells) == 0:
        logger.error("Validation failed: no code cells found")
        return False, "Notebook contains no code cells", None
 
    logger.info(f"[ATEFEH] Input validation successful | code_cells={len(code_cells)}")
    return True, "Valid", nb
 
logger.info("validate_input ready")

2026-07-19 18:29:22,944 - INFO - validate_input ready


<h3 style="color:#2C3E50;">
Cell 7 — execute_notebook (ATEFEH-Agent · Step 4a)
</h3>
<p style="font-size:16px; line-height:1.9; text-align:justify;">
This function is responsible for reading and preparing the target Jupyter
notebook for static analysis. Rather than running the notebook in a live
kernel, it preserves the notebook object exactly as submitted — including
any outputs already stored inside the .ipynb file from its original run
(e.g. on Kaggle) — and saves a working copy for the downstream steps.
This design allows the ATEFEH pipeline to explain a notebook's task,
logic, and structure without requiring access to external datasets.
</p>
<ul style="font-size:16px; line-height:1.9;">
<li>
<b>Input:</b> <code>nb (NotebookNode)</code> — parsed notebook object returned from <code>validate_input</code>
</li>
<li>
<b>Input:</b> <code>notebook_path (str)</code> — original file path, used to resolve the notebook's stem for saving a copy
</li>
<li>
<b>Input:</b> <code>timeout (int)</code> — retained for interface compatibility; unused in static mode
</li>
<li>
<b>Output:</b> <code>nb (NotebookNode)</code> — the same notebook object, unmodified, ready for source and output extraction
</li>
</ul>
<div style="
background-color:#F8F9F9;
padding:14px;
border-left:5px solid #5DADE2;
border-radius:8px;
margin-top:15px;">
<b>Important:</b>
No code is executed in this step. The notebook's structure and any
pre-existing outputs are read as-is, so the pipeline never depends on
external datasets being present on the local machine.
</div>

In [48]:
def execute_notebook(nb, notebook_path: str, timeout: int = 120):
    """
    Static mode (ATEFEH-Agent - Step 4a).
    Does not execute the notebook. Returns the notebook object as-is,
    including any outputs already stored inside the .ipynb file
    (e.g. from when it was originally run on Kaggle).
    """
    PIPELINE_FILENAME = "atefeh-pipeline.ipynb"
    if Path(notebook_path).name == PIPELINE_FILENAME:
        logger.error(
            "[ATEFEH] Refusing to process the pipeline notebook on itself "
            "(self-reference). Point NOTEBOOK_PATH / NOTEBOOK_LIST at a "
            "target ML notebook instead."
        )
        raise ValueError("Self-reference detected: cannot process the pipeline on itself.")

    logger.info("[ATEFEH] Static mode: skipping real execution")               # confirmation 1
    logger.info(f"[ATEFEH] Reading notebook structure from: {notebook_path}")  # confirmation 2

    try:
        executed_path = OUTPUT_DIR / f"{Path(notebook_path).stem}_executed.ipynb"
        with open(executed_path, "w", encoding="utf-8") as f:
            nbformat.write(nb, f)
        logger.info(f"[ATEFEH] Notebook copy saved: {executed_path}")         # confirmation 3
    except Exception as save_error:
        logger.error(f"[ATEFEH] Failed to save notebook copy: {save_error}")

    return nb

logger.info("execute_notebook ready (static mode)")   # printed immediately when this cell runs

2026-07-19 20:15:32,170 - INFO - execute_notebook ready (static mode)


<h3 style="color:#2C3E50;">
Cell 8 — extract_outputs (ATEFEH-Agent · Step 4b)
</h3>
<p style="font-size:16px; line-height:1.9; text-align:justify;">
This function is responsible for extracting and structuring the source
code of each cell, together with any outputs that were already stored
inside the .ipynb file from the notebook's original run (e.g. on Kaggle).
It processes each code cell individually and converts its source and any
saved Jupyter outputs into a standardized, machine-readable format.
The extracted data includes source code, saved streams, saved execution
results, saved display data (including visualizations), and saved error
traces, enabling downstream explainable AI summarization — all without
re-executing the notebook.
</p>
<ul style="font-size:16px; line-height:1.9;">
<li>
<b>Input:</b> <code>nb (NotebookNode)</code> — notebook object returned from <code>execute_notebook</code>
</li>
<li>
<b>Output:</b> <code>list of dicts</code> — structured representation of each code cell, its source code, and any pre-existing outputs
</li>
</ul>
<div style="
background-color:#F8F9F9;
padding:14px;
border-left:5px solid #5DADE2;
border-radius:8px;
margin-top:15px;">
<b>Note:</b>
Each dictionary corresponds to a single code cell and contains both the
original source code (truncated if necessary) and any outputs that were
already present in the notebook file — no code is run in this step.
</div>

In [49]:
def extract_outputs(nb) -> list:
    """
    Static mode (ATEFEH-Agent - Step 4b).
    For each code cell, returns the full source code plus any outputs
    already stored in the .ipynb file. No execution is performed here.
    """
    logger.info("[ATEFEH] Extracting cell sources (static mode)")   # confirmation 1
    results = []

    for idx, cell in enumerate(nb.cells):
        if cell.cell_type != "code":
            continue

        cell_data = {
            "cell_number": idx + 1,
            "source": cell.source[:500],
            "outputs": []
        }

        for output in cell.get("outputs", []):
            if output.output_type == "stream":
                cell_data["outputs"].append({"type": "stream", "text": "".join(output.get("text", ""))})
            elif output.output_type == "execute_result":
                cell_data["outputs"].append({"type": "execute_result", "data": output.get("data", {}).get("text/plain", "")})
            elif output.output_type == "display_data":
                cell_data["outputs"].append({"type": "display_data", "has_plot": "image/png" in output.get("data", {}), "text": output.get("data", {}).get("text/plain", "")})
            elif output.output_type == "error":
                cell_data["outputs"].append({"type": "error", "ename": output.get("ename", ""), "evalue": output.get("evalue", "")})

        results.append(cell_data)

    logger.info(f"[ATEFEH] Extraction complete | cells={len(results)}")   # confirmation 2 (cell count)
    return results

logger.info("extract_outputs ready (static mode)")   # printed immediately when this cell runs

2026-07-19 20:16:28,028 - INFO - extract_outputs ready (static mode)


<h3 style="color:#2C3E50;">
Cell 9 — save_outputs / load_outputs (ATEFEH-Agent · Step 4c helpers)
</h3>

<p style="font-size:16px; line-height:1.9; text-align:justify;">

This section provides utility functions for persisting and reloading extracted notebook outputs. These helpers are used to store intermediate results of the ATEFEH pipeline and enable reproducible execution of the LLM summarisation stage without re-running the full notebook.

Saving outputs to JSON allows the pipeline to decouple execution from explanation generation, improving efficiency and debugging capability.

</p>

<h4 style="color:#34495E; margin-top:15px;">
save_outputs
</h4>

<ul style="font-size:16px; line-height:1.9;">

<li>
<b>Input:</b> <code>cell_outputs (list)</code> — structured output list produced by <code>extract_outputs</code>
</li>

<li>
<b>Input:</b> <code>title (str)</code> — notebook stem used to generate the output filename
</li>

<li>
<b>Output:</b> <code>out_file (Path)</code> — file path of the saved JSON output
</li>

</ul>

<h4 style="color:#34495E; margin-top:15px;">
load_outputs
</h4>

<ul style="font-size:16px; line-height:1.9;">

<li>
<b>Input:</b> <code>title (str)</code> — notebook stem corresponding to a previously saved output file
</li>

<li>
<b>Output:</b> <code>data (list)</code> — deserialized list of structured cell outputs
</li>

</ul>

<div style="
background-color:#F8F9F9;
padding:14px;
border-left:5px solid #5DADE2;
border-radius:8px;
margin-top:15px;">

<b>Why JSON Storage?</b><br><br>

Saving extracted outputs enables the LLM-Agent stage to be executed independently from notebook execution. This is particularly useful for:
<ul>
<li>Prompt engineering and iterative refinement</li>
<li>Debugging explanation generation</li>
<li>Reducing redundant computation during development</li>
</ul>

</div>

In [34]:
def save_outputs(cell_outputs: list, title: str) -> Path:
    """
    Persist extracted cell outputs to a JSON file for later reuse (ATEFEH-Agent · Step 4c helper).
 
    Saving the outputs separately from the full executed notebook allows the
    LLM step (generate_explanation) to be re-run in isolation without
    re-executing the notebook — useful when tuning the prompt or debugging.
 
    Parameters
    ----------
    cell_outputs : list of dict
        The list returned by extract_outputs.
        Each dict contains "cell_number" (int), "source" (str), and
        "outputs" (list of typed output dicts). Must be JSON-serialisable;
        any non-serialisable values (e.g. Path objects) are converted to
        strings via json.dump's default=str argument.
 
    title : str
        The notebook stem — filename without extension.
        Produced by submit_ml_program and threaded through the pipeline.
        Used to construct the output filename:
            OUTPUT_DIR/<title>_outputs.json
        Example: title="heart_disease_predictions" →
                 "notebook_outputs/heart_disease_predictions_outputs.json"
 
    Returns
    -------
    pathlib.Path
        The Path object pointing to the written JSON file.
        Example: PosixPath('notebook_outputs/heart_disease_predictions_outputs.json')
        This path is stored in the final result dict under "outputs_file"
        and can be passed to load_outputs to reload the data without
        re-executing the notebook.
    """
    out_file = OUTPUT_DIR / f"{title}_outputs.json"
    with open(out_file, "w", encoding="utf-8") as f:
        # indent=2    → human-readable formatting
        # default=str → converts non-serialisable objects (e.g. Path) to strings
        # ensure_ascii=False → preserves unicode characters correctly
        json.dump(cell_outputs, f, indent=2, ensure_ascii=False, default=str)
    logger.info(f"[ATEFEH] Outputs saved: {out_file}")
    return out_file
 
 
def load_outputs(title: str) -> list:
    """
    Reload cell outputs that were previously saved by save_outputs.
 
    Use this to re-run the LLM step without re-executing the notebook.
    The returned list is identical in structure to what extract_outputs
    originally produced.
 
    Parameters
    ----------
    title : str
        The notebook stem used when save_outputs was originally called.
        Used to reconstruct the expected filename:
            OUTPUT_DIR/<title>_outputs.json
        Example: "heart_disease_predictions" →
                 "notebook_outputs/heart_disease_predictions_outputs.json"
        Must match exactly — no fuzzy matching is performed.
 
    Returns
    -------
    list of dict
        The deserialized contents of OUTPUT_DIR/<title>_outputs.json.
        Identical in structure to the list returned by extract_outputs:
        one dict per code cell, each with keys:
            "cell_number" (int), "source" (str), "outputs" (list of dict).
        Can be passed directly to build_xai_summary.
 
    Raises
    ------
    FileNotFoundError
        If OUTPUT_DIR/<title>_outputs.json does not exist.
        This means save_outputs has not yet been run for this title,
        or the outputs directory was cleared. Run the full pipeline first.
    """
    out_file = OUTPUT_DIR / f"{title}_outputs.json"
 
    # --- Guard: raise a clear error if the file doesn't exist yet ---
    if not out_file.exists():
        logger.error(f"Output file not found: {out_file}")
        raise FileNotFoundError(f"No saved outputs for: {title}")
 
    with open(out_file, "r", encoding="utf-8") as f:
        data = json.load(f)   # deserialize JSON → Python list of dicts
 
    logger.info(
        f"[ATEFEH] Outputs loaded: {out_file} | cells={len(data)}")
    return data
 
logger.info("save_outputs / load_outputs ready")

2026-07-19 18:29:34,881 - INFO - save_outputs / load_outputs ready


<h3 style="color:#2C3E50;">
Cell 10 — build_xai_summary (ATEFEH-Agent · Step 4c)
</h3>
<p style="font-size:16px; line-height:1.9; text-align:justify;">
This function constructs a structured textual summary from each cell's
source code and any outputs already stored in the notebook file. The
resulting summary serves as the primary input to the LLM-Agent and is
designed to be compact, informative, and suitable for prompt injection
into a large language model.
It aggregates source code, saved outputs, saved errors, and saved
visualisation indicators into a single coherent representation of the
notebook's logic and structure — without executing any code.
</p>
<ul style="font-size:16px; line-height:1.9;">
<li>
<b>Input:</b> <code>cell_outputs (list)</code> — structured list returned by <code>extract_outputs</code>
</li>
<li>
<b>Input:</b> <code>title (str)</code> — notebook stem used in the summary header
</li>
<li>
<b>Output:</b> <code>summary (str)</code> — plain-text representation of the notebook's code and stored outputs, suitable for LLM prompting
</li>
</ul>
<div style="
background-color:#F8F9F9;
padding:14px;
border-left:5px solid #5DADE2;
border-radius:8px;
margin-top:15px;">
<b>Purpose:</b><br>
The generated XAI summary acts as a compressed semantic representation of
the notebook's code, enabling the LLM-Agent to reason about the model's
task, logic, and limitations from source code and any pre-existing
outputs — without requiring the notebook to be executed or the original
dataset to be present.
</div>

In [50]:
def build_xai_summary(cell_outputs: list, title: str) -> str:
    """
    Builds a plain-text XAI summary from cell source code and any
    previously saved outputs (static mode - no execution performed).
    """
    logger.info("[ATEFEH] Building XAI summary (static mode)")

    lines = [f"ML Notebook: {title}", "=" * 50, ""]
    included = 0

    for cell in cell_outputs:
        lines.append(f"[Cell {cell['cell_number']}]")
        lines.append(f"Code: {cell['source']}")

        for out in cell["outputs"]:
            if out["type"] == "stream" and out["text"].strip():
                lines.append(f"Saved Output : {out['text'][:400]}")
            elif out["type"] == "execute_result" and out["data"].strip():
                lines.append(f"Saved Result : {out['data'][:400]}")
            elif out["type"] == "display_data":
                lines.append("[Saved Visualisation / plot]")
            elif out["type"] == "error":
                lines.append(f"Saved Error : {out['ename']}: {out['evalue']}")

        lines.append("")
        included += 1

    logger.info(f"[ATEFEH] XAI summary built | cells={included}")
    return "\n".join(lines)
logger.info("build_xai_summary ready (static mode)")

2026-07-19 20:17:10,195 - INFO - build_xai_summary ready (static mode)


<h2 style="color:#2C3E50;">
LLM-Agent
</h2>

<p style="font-size:16px; line-height:1.9; text-align:justify;">

The LLM-Agent is responsible for generating the final natural-language explanation of the executed notebook. It interfaces with a local large language model (Ollama), constructs structured prompts from the XAI summary, evaluates response quality, and ensures the final output meets academic-level standards.

</p>

<ul style="font-size:16px; line-height:1.9;">

<li>
<b>Step 5 — Model Interaction:</b> verifies Ollama availability, builds prompts, queries the LLM, and validates response quality.
</li>

</ul>

<hr style="margin-top:15px; margin-bottom:15px;">

<h3 style="color:#34495E;">
Cell 11 — check_ollama (LLM-Agent · utility)
</h3>

<p style="font-size:16px; line-height:1.9; text-align:justify;">

This utility function checks whether the local Ollama server is running and reachable. It performs a simple HTTP request to the base URL and confirms that the service is available before any LLM calls are made.

</p>

<ul style="font-size:16px; line-height:1.9;">

<li>
<b>Input:</b> <code>url (str)</code> — base URL of the Ollama server (default: http://localhost:11434)
</li>

<li>
<b>Output:</b> <code>bool</code> — returns True if the server responds with HTTP status 200, otherwise False
</li>

</ul>

<div style="
background-color:#F8F9F9;
padding:14px;
border-left:5px solid #5DADE2;
border-radius:8px;
margin-top:15px;">

<b>Note:</b>
This step ensures that the LLM backend is available before attempting any prompt generation or inference calls.
</div>

In [36]:
def check_ollama(url: str = "http://localhost:11434") -> bool:
    """
    Ping the local Ollama server (LLM-Agent · utility).
 
    Parameters
    ----------
    url : str, optional
        Base URL. Default: "http://localhost:11434"
        Do NOT include a path — just scheme + host + port.
        Custom example: "http://192.168.1.50:11434"
 
    Returns
    -------
    bool
        True  — HTTP 200 received within 3 seconds.
        False — any of: non-200 status, connection refused,
                timeout, or network error. Fix: run 'ollama serve'.
    """
    try:
        response = requests.get(url, timeout=3)  # 3s health-check timeout
        if response.status_code == 200:
            logger.info("[LLM-Agent] Ollama server reachable")
            return True
        logger.warning(f"[LLM-Agent] Ollama responded with status {response.status_code}")
        return False
    except Exception as e:
        logger.error(f"[LLM-Agent] Ollama not reachable: {e}\n  Fix: run 'ollama serve'")
        return False
 
logger.info("check_ollama ready")
 

2026-07-19 18:29:38,329 - INFO - check_ollama ready


<h3 style="color:#2C3E50;">
Cell 12 — build_prompt (LLM-Agent · utility)
</h3>

<p style="font-size:16px; line-height:1.9; text-align:justify;">

This function constructs the complete prompt that is sent to the Ollama large language model. It embeds the structured XAI summary into a carefully designed instruction template that guides the model to produce a structured academic explanation of the notebook.

The prompt defines the expected output format, including task description, results, behaviour analysis, and limitations. It may also include additional retry-specific instructions when multiple attempts are required.

</p>

<ul style="font-size:16px; line-height:1.9;">

<li>
<b>Input:</b> <code>xai_summary (str)</code> — structured plain-text output from <code>build_xai_summary</code>
</li>

<li>
<b>Input:</b> <code>attempt (int)</code> — current retry iteration (1-based index)
</li>

<li>
<b>Output:</b> <code>prompt (str)</code> — fully constructed prompt string ready to be sent to Ollama
</li>

</ul>

<div style="
background-color:#F8F9F9;
padding:14px;
border-left:5px solid #5DADE2;
border-radius:8px;
margin-top:15px;">

<b>Purpose:</b><br>
This function ensures that the LLM receives a consistent and structured instruction format, improving the reliability and academic quality of the generated explanation.
</div>

In [37]:
def build_prompt(xai_summary: str, attempt: int = 1) -> str:
    """
    Build the complete LLM prompt (LLM-Agent · utility).
 
    Parameters
    ----------
    xai_summary : str
        From build_xai_summary. Embedded under "NOTEBOOK OUTPUTS:" header.
        Typically 500–3000 chars; affects total prompt length.
    attempt : int, optional
        1-based retry counter (default 1).
        attempt == 1: base prompt returned as-is.
        attempt > 1:  IMPORTANT note appended requesting a better response.
 
    Returns
    -------
    str
        Complete prompt with structure:
            "You are ATEFEH, an XAI assistant..."
            "NOTEBOOK OUTPUTS:"  + xai_summary
            "1. TASK / 2. RESULTS / 3. BEHAVIOUR / 4. LIMITATIONS"
            [if attempt > 1]: "IMPORTANT: attempt N, previous too short..."
        Typical length: 800–4000 chars.
    """
    prompt = f"""You are ATEFEH, an Explainable AI (XAI) assistant for machine learning research.
 
Analyse the following ML notebook outputs and produce a structured explanation suitable for an academic dissertation.
 
NOTEBOOK OUTPUTS:
{xai_summary}
 
Your explanation must cover these four sections:
1. TASK        -- What ML task does this notebook address?
2. RESULTS     -- What are the key numerical outputs and metrics?
3. BEHAVIOUR   -- How does the model behave? Any notable patterns?
4. LIMITATIONS -- Any warnings, errors, or methodological limits?
 
Write at a technical level appropriate for a PhD dissertation."""
 
    if attempt > 1:
        prompt += (
            f"\n\nIMPORTANT: This is attempt {attempt}. "
            "The previous response was too short or incomplete. "
            "Please be more specific and cover all four sections."
        )
 
    logger.info(f"[LLM-Agent] Prompt built | attempt={attempt} | length={len(prompt)}")
    return prompt
 
logger.info("build_prompt ready")
 

2026-07-19 18:29:39,726 - INFO - build_prompt ready


<h3 style="color:#2C3E50;">
Cell 13 — is_satisfactory (LLM-Agent · quality gate)
</h3>

<p style="font-size:16px; line-height:1.9; text-align:justify;">

This function acts as a quality control gate for the LLM output. It evaluates the generated explanation and determines whether it meets minimum structural and semantic standards before being accepted by the pipeline.

The goal is to ensure that the final explanation is sufficiently detailed, complete, and covers the required analytical dimensions.

</p>

<ul style="font-size:16px; line-height:1.9;">

<li>
<b>Input:</b> <code>response (str)</code> — raw natural-language output returned by the LLM-Agent
</li>

<li>
<b>Output:</b> <code>bool</code> — returns True only if all defined quality criteria are satisfied
</li>

</ul>

<div style="
background-color:#F8F9F9;
padding:14px;
border-left:5px solid #5DADE2;
border-radius:8px;
margin-top:15px;">

<b>Purpose:</b><br>
This gate ensures that low-quality or incomplete LLM responses are filtered out before being accepted by the pipeline, enforcing consistency and academic-level completeness in generated explanations.
</div>

In [38]:
def is_satisfactory(response: str) -> bool:
    """
    Quality gate for the LLM Loop (LLM-Agent · utility).
 
    Parameters
    ----------
    response: str
        Raw text from response.json().get("response", "").
        Whitespace stripped before length check; case-insensitive for keywords.
 
    Returns
    -------
    bool
        True  — both criteria pass:
                  len(response.strip()) >= 150
                  AND >= 2 of 5 keywords found
        False — criterion 1 failed: response too short (< 150 chars)
        False — criterion 2 failed: < 2 keywords found
                Keywords checked: "task", "result", "model", "behaviour", "limitation"
                Note: criterion 1 evaluated first; criterion 2 skipped if 1 fails.
    """
    if len(response.strip()) < 150:
        logger.warning("[LLM-Agent] Quality check failed: response too short -- retrying")
        return False
 
    required_keywords = ["task", "result", "model", "behaviour", "limitation"]
    found = sum(1 for kw in required_keywords if kw.lower() in response.lower())
 
    if found < 2:
        logger.warning(f"[LLM-Agent] Quality check failed: only {found}/5 keywords found -- retrying")
        return False
 
    logger.info(f"[LLM-Agent] Quality check passed | length={len(response)} | sections={found}/5")
    return True
 
logger.info("is_satisfactory ready")

2026-07-19 18:29:41,093 - INFO - is_satisfactory ready


<h3 style="color:#2C3E50;">
Cell 14 — generate_explanation / LLM Loop (LLM-Agent · Step 5)
</h3>

<p style="font-size:16px; line-height:1.9; text-align:justify;">

This function implements the core LLM inference loop of the ATEFEH pipeline. It is responsible for communicating with the Ollama model, generating a natural-language explanation from the structured XAI summary, and ensuring output quality through iterative retries.

The loop continues until a satisfactory response is produced or the maximum number of attempts is reached. This mechanism improves robustness against incomplete or low-quality model outputs.

</p>

<ul style="font-size:16px; line-height:1.9;">

<li>
<b>Input:</b> <code>xai_summary (str)</code> — structured summary generated by <code>build_xai_summary</code>
</li>

<li>
<b>Input:</b> <code>max_retries (int)</code> — maximum number of LLM inference attempts (default: 3)
</li>

<li>
<b>Input:</b> <code>timeout (int)</code> — request timeout in seconds for each API call (default: 120)
</li>

<li>
<b>Output:</b> <code>explanation (str)</code> — final natural-language explanation, or an error message if the Ollama service is unreachable
</li>

</ul>

<div style="
background-color:#F8F9F9;
padding:14px;
border-left:5px solid #5DADE2;
border-radius:8px;
margin-top:15px;">

<b>Important:</b><br>
If the Ollama server is not running or cannot be reached, the function returns a descriptive error string instead of raising a crash, ensuring graceful failure of the pipeline.
</div>

In [39]:
def generate_explanation(xai_summary: str, max_retries: int = 3, timeout: int = 120) -> str:
    """
    Call Ollama and return a quality-checked explanation (LLM-Agent · Step 5).
 
    Parameters
    ----------
    xai_summary : str
        From build_xai_summary. Passed to build_prompt unchanged.
    max_retries : int, optional
        Max attempts (default 3). Increase to 5 if model is inconsistent.
        Early exit on first is_satisfactory() pass.
    timeout : int, optional
        Per-request HTTP timeout in seconds (default 120).
        Increase for large models: llama3:70b on CPU needs 600+.
 
    Returns
    -------
    str
        On success      : full explanation with TASK/RESULTS/BEHAVIOUR/LIMITATIONS
        On unreachable  : "[LLM-Agent] Ollama is not running.\nRun 'ollama serve'..."
        On all retries  : last response received (may be short/incomplete)
                          empty string "" if every attempt raised an exception
    """
    logger.info("[LLM-Agent] LLM Loop started")
 
    if not check_ollama():
        return "[LLM-Agent] Ollama is not running.\nRun 'ollama serve' then 'ollama pull llama3'."
 
    last_response = ""  # fallback: returned if all attempts fail the quality gate
 
    for attempt in range(1, max_retries + 1):
        logger.info(f"[LLM-Agent] LLM Loop attempt {attempt}/{max_retries}")
        prompt = build_prompt(xai_summary, attempt)
 
        try:
            response = requests.post(
                OLLAMA_URL,
                json={
                    "model" : OLLAMA_MODEL,
                    "prompt": prompt,
                    "stream": False   # CRITICAL: False -> full JSON in one go; True -> json() fails
                },
                timeout=timeout
            )
            response.raise_for_status()
 
            explanation   = response.json().get("response", "")
            last_response = explanation
 
            logger.info(f"[LLM-Agent] Response received | length={len(explanation)}")
 
            if is_satisfactory(explanation):
                logger.info(f"[LLM-Agent] LLM Loop completed | attempt={attempt}")
                return explanation  # early exit — no need to retry
 
            logger.info("[LLM-Agent] Response below quality threshold -- retrying...")
 
        except requests.exceptions.Timeout:
            logger.error(f"[LLM-Agent] Request timed out at attempt {attempt}")
        except Exception as e:
            logger.exception(f"[LLM-Agent] Unexpected error at attempt {attempt}: {e}")
 
    logger.warning("[LLM-Agent] Max retries reached; returning last response")
    return last_response
 
logger.info("generate_explanation ready")
 

2026-07-19 18:29:42,684 - INFO - generate_explanation ready


<h2 style="color:#2C3E50;">
PIPELINE Controller
</h2>

<p style="font-size:16px; line-height:1.9; text-align:justify;">

The PIPELINE Controller acts as the central orchestrator of the entire ATEFEH system. It coordinates all agents in a fixed execution sequence, ensuring that each stage of the pipeline is executed correctly and that intermediate outputs are passed between components in a structured manner.

This controller integrates all previous modules into a single end-to-end workflow, from notebook submission to final explanation generation.

</p>

<hr style="margin-top:15px; margin-bottom:15px;">

<h3 style="color:#34495E;">
Cell 15 — run_full_pipeline (Orchestrator)
</h3>

<p style="font-size:16px; line-height:1.9; text-align:justify;">

This function represents the main entry point of the ATEFEH pipeline. It sequentially executes all stages of the system, including validation, execution, output extraction, summarisation, and LLM-based explanation generation.

It ensures proper coordination between the User-Agent, ATEFEH-Agent, and LLM-Agent, and returns a structured result dictionary containing the final output and metadata.

</p>

<ul style="font-size:16px; line-height:1.9;">

<li>
<b>Input:</b> <code>notebook_path (str)</code> — file-system path to the target <code>.ipynb</code> notebook
</li>

<li>
<b>Output:</b> <code>result (dict)</code> — structured pipeline output including status, extracted cells, explanation, and saved artefacts
</li>

</ul>

<div style="
background-color:#F8F9F9;
padding:14px;
border-left:5px solid #5DADE2;
border-radius:8px;
margin-top:15px;">

<b>Important:</b><br>
This function is the central orchestrator of the ATEFEH architecture and must be executed after all dependent components are properly defined and initialized.
</div>

In [40]:
def run_full_pipeline(notebook_path: str) -> dict:
    """
    Execute the complete ATEFEH pipeline for a single notebook (Orchestrator).
 
    Parameters
    ----------
    notebook_path : str
        Path to the .ipynb notebook. e.g. "models/heart_disease_predictions.ipynb"
 
    Returns
    -------
    dict
        On validation failure:
            "status"  : "error"
            "message" : one of the four validate_input error messages
 
        On success:
            "status"          : "success"
            "notebook"        : str  — notebook stem e.g. "heart_disease_predictions"
            "cells_extracted" : int  — number of code cells processed e.g. 12
            "explanation"     : str  — LLM text (500–3000 chars, 4 sections)
            "outputs_file"    : str  — path to saved JSON e.g. "notebook_outputs/..."
 
        Side effects (files written to disk):
            OUTPUT_DIR/<title>_executed.ipynb
            OUTPUT_DIR/<title>_outputs.json
            OUTPUT_DIR/<title>_explanation.txt
            OUTPUT_DIR/<title>_atefeh_result.json
    """
    logger.info("=" * 60)
    logger.info("ATEFEH pipeline started")
    logger.info("=" * 60)
 
    logger.info("[Step 1] User-Agent: Submit ML Program")
    submission = submit_ml_program(notebook_path)
    title      = submission["title"]
 
    logger.info("[Step 2] ATEFEH-Agent: Validate")
    is_valid, message, nb = validate_input(submission)
    if not is_valid:
        logger.error(f"Pipeline stopped: {message}")
        return {"status": "error", "message": message}
 
    logger.info("[Step 4a] ATEFEH-Agent: Execute notebook")
    nb_executed = execute_notebook(nb, notebook_path)
 
    logger.info("[Step 4b] ATEFEH-Agent: Extract outputs")
    cell_outputs = extract_outputs(nb_executed)
 
    logger.info("[Step 4c] ATEFEH-Agent: Save outputs and build summary")
    outputs_file = save_outputs(cell_outputs, title)
    xai_summary  = build_xai_summary(cell_outputs, title)
 
    logger.info("[Step 5] LLM-Agent: Generate explanation (LLM Loop)")
    explanation = generate_explanation(xai_summary)
 
    logger.info("[Step 6] User-Agent: Send explanation")
    receive_explanation(explanation, title)
 
    result = {
        "status"          : "success",
        "notebook"        : title,
        "cells_extracted" : len(cell_outputs),
        "explanation"     : explanation,
        "outputs_file"    : str(outputs_file)
    }
 
    result_file = OUTPUT_DIR / f"{title}_atefeh_result.json"
    with open(result_file, "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2, ensure_ascii=False)
 
    logger.info("=" * 60)
    logger.info(f"ATEFEH pipeline completed | notebook={title} | cells={len(cell_outputs)}")
    logger.info("=" * 60)
 
    return result
 
logger.info("run_full_pipeline ready")

2026-07-19 18:29:44,282 - INFO - run_full_pipeline ready


<h3 style="color:#2C3E50;">
Cell 16 — Run Single Notebook
</h3>

<p style="font-size:16px; line-height:1.9; text-align:justify;">

This cell executes the complete ATEFEH pipeline for a single Jupyter notebook defined in the configuration section (NOTEBOOK_PATH). It serves as a simple entry point for running the system without requiring batch processing.

The execution follows the full pipeline workflow, including validation, execution, output extraction, summarisation, and LLM-based explanation generation.

</p>

<div style="
background-color:#F8F9F9;
padding:14px;
border-left:5px solid #5DADE2;
border-radius:8px;
margin-top:15px;">

<b>Note:</b><br>
To process multiple notebooks in a batch mode, use Cell 17 instead of this cell.
</div>

In [41]:
result = run_full_pipeline(NOTEBOOK_PATH)
logger.info(f"Pipeline execution status: {result['status']}")

2026-07-19 18:29:46,922 - INFO - ============================================================
2026-07-19 18:29:46,927 - INFO - ATEFEH pipeline started
2026-07-19 18:29:46,930 - INFO - ============================================================
2026-07-19 18:29:46,934 - INFO - [Step 1] User-Agent: Submit ML Program
2026-07-19 18:29:46,936 - INFO - [User-Agent] Submitting notebook: Dataset.ipynb
2026-07-19 18:29:46,942 - INFO - [User-Agent] Submission ready | title=Dataset | size=9.2 KB
2026-07-19 18:29:46,944 - INFO - [Step 2] ATEFEH-Agent: Validate
2026-07-19 18:29:46,948 - INFO - [ATEFEH] Starting input validation
2026-07-19 18:29:46,964 - INFO - [ATEFEH] Input validation successful | code_cells=5
2026-07-19 18:29:46,970 - INFO - [Step 4a] ATEFEH-Agent: Execute notebook
2026-07-19 18:29:46,979 - INFO - [ATEFEH] Static mode: skipping real execution
2026-07-19 18:29:46,981 - INFO - [ATEFEH] Reading notebook structure from: Dataset.ipynb
2026-07-19 18:29:47,005 - INFO - [ATEFEH] Noteboo

<h2 style="color:#2C3E50;">
Cell 17 — Run All Notebooks (Batch Loop)
</h2>

<p style="font-size:16px; line-height:1.9; text-align:justify;">

This cell executes the ATEFEH pipeline in batch mode over multiple Jupyter notebooks. Each notebook is processed independently through the full pipeline, including validation, execution, output extraction, summarization, and LLM-based explanation generation.

</p>

<h3 style="color:#34495E;">Input</h3>

<ul style="font-size:16px; line-height:1.9;">
<li><b>notebook_list (list of dicts)</b> — each dictionary contains:
    <ul>
        <li><code>path</code>: file path to a .ipynb notebook</li>
        <li><code>task</code>: machine learning task label (e.g., regression, classification)</li>
    </ul>
</li>
</ul>

<h3 style="color:#34495E;">Output</h3>

<ul style="font-size:16px; line-height:1.9;">
<li><b>all_results (list of dicts)</b> — one result per notebook containing:
    <ul>
        <li><code>status</code>: success / skipped / error</li>
        <li><code>notebook</code>: notebook name (stem)</li>
        <li><code>task</code>: ML task type</li>
        <li>All outputs from <code>run_full_pipeline</code> for successful runs</li>
    </ul>
</li>
</ul>

<h3 style="color:#34495E;">Design Decisions</h3>

<div style="
background-color:#F8F9F9;
padding:14px;
border-left:5px solid #5DADE2;
border-radius:8px;
margin-top:10px;
font-size:15px;
line-height:1.8;">

<ul>
<li>Missing notebooks are skipped instead of stopping execution</li>
<li>Each notebook is executed independently to avoid cascading failures</li>
<li>Exceptions are handled per notebook to ensure batch robustness</li>
<li>Final results are saved as <code>all_results.json</code> in OUTPUT_DIR</li>
</ul>

</div>

In [42]:
def run_all_notebooks(notebook_list: list) -> list:
    total = len(notebook_list)
    all_results = []

    logger.info(f"ATEFEH batch started | total={total}")

    for idx, entry in enumerate(notebook_list, start=1):
        nb_path = entry["path"]
        nb_task = entry.get("task", "unknown")

        logger.info(f"[{idx}/{total}] Processing: {nb_path} | task: {nb_task}")

        if not os.path.exists(nb_path):
            logger.error(f"Skipping: file not found -- {nb_path}")
            all_results.append({
                "status": "skipped",
                "notebook": Path(nb_path).stem,
                "task": nb_task,
                "message": "File not found"
            })
            continue

        try:
            result = run_full_pipeline(nb_path)
            result["task"] = nb_task
            all_results.append(result)

        except Exception as e:
            logger.exception(f"[{idx}/{total}] Failed: {nb_path} | error={e}")
            all_results.append({
                "status": "error",
                "notebook": Path(nb_path).stem,
                "task": nb_task,
                "message": str(e)
            })

    all_results_file = OUTPUT_DIR / "all_results.json"
    with open(all_results_file, "w", encoding="utf-8") as f:
        json.dump(all_results, f, indent=2, ensure_ascii=False)

    success = sum(1 for r in all_results if r["status"] == "success")
    skipped = sum(1 for r in all_results if r["status"] == "skipped")
    errors = sum(1 for r in all_results if r["status"] == "error")

    logger.info(f"Batch completed | success={success} skipped={skipped} errors={errors}")

    return all_results

In [43]:
NOTEBOOK_LIST = [
    # --- Supervised: Regression ---
    {"path": "kaggle_notebooks/nyc_taxi_fare_prediction.ipynb", "task": "regression"},
    {"path": "kaggle_notebooks/tps_feb_2021_lgbmregressor.ipynb", "task": "regression"},
    {"path": "kaggle_notebooks/concrete_compressive_strength.ipynb", "task": "regression"},
    {"path": "kaggle_notebooks/diamond_price_prediction.ipynb", "task": "regression"},
    {"path": "kaggle_notebooks/salary_prediction.ipynb", "task": "regression"},
    {"path": "kaggle_notebooks/bike_demand_forecasting.ipynb", "task": "regression"},

    # --- Supervised: Binary Classification ---
    {"path": "kaggle_notebooks/heart_disease_predictions.ipynb", "task": "binary_classification"},
    {"path": "kaggle_notebooks/lending_club_loan_defaulters.ipynb", "task": "binary_classification"},
    {"path": "kaggle_notebooks/bank_customer_churn_prediction.ipynb", "task": "binary_classification"},
    {"path": "kaggle_notebooks/credit_fraud_imbalanced.ipynb", "task": "binary_classification"},
    {"path": "kaggle_notebooks/fake_news_detection.ipynb", "task": "binary_classification"},

    # --- Supervised: Multi-class Classification ---
    {"path": "kaggle_notebooks/movie_genre_classification.ipynb", "task": "multiclass_classification"},
    {"path": "kaggle_notebooks/resumesorter_nlp_classification.ipynb", "task": "multiclass_classification"},
    {"path": "kaggle_notebooks/reddit_title_to_subreddit.ipynb", "task": "multiclass_classification"},
    {"path": "kaggle_notebooks/random_forest_text_classification.ipynb", "task": "multiclass_classification"},
    {"path": "kaggle_notebooks/nifty50_stock_prediction.ipynb", "task": "multiclass_classification"},
    {"path": "kaggle_notebooks/wine_quality_prediction.ipynb", "task": "multiclass_classification"},
    {"path": "kaggle_notebooks/language_detection.ipynb", "task": "multiclass_classification"},

    # --- Deep Learning ---
    {"path": "kaggle_notebooks/book_genre_classification_goodreads.ipynb", "task": "deep_learning"},
    {"path": "kaggle_notebooks/bert_base_uncased_pytorch.ipynb", "task": "deep_learning"},
    {"path": "kaggle_notebooks/vehicle_classification.ipynb", "task": "deep_learning"},
    {"path": "kaggle_notebooks/20_news_groups_cnn.ipynb", "task": "deep_learning"},
    
    # --- Semi-Supervised ---
    {"path": "kaggle_notebooks/semi_supervised_text_classification.ipynb", "task": "semi_supervised"},
    {"path": "kaggle_notebooks/semi_supervised_document_classification.ipynb", "task": "semi_supervised"},

    # --- Unsupervised ---
    
    {"path": "kaggle_notebooks/anomaly_detection_isolation_forest.ipynb", "task": "unsupervised_anomaly_detection"},
    {"path": "kaggle_notebooks/surveillance_system_anomaly_detection.ipynb", "task": "unsupervised_anomaly_detection"},
    {"path": "kaggle_notebooks/anomaly_detection_multivariate_timeseries.ipynb", "task": "unsupervised_anomaly_detection"},
]


all_results = run_all_notebooks(NOTEBOOK_LIST)
logger.info(f"Batch execution status: {len(all_results)} notebooks processed")



def execute_notebook(nb, notebook_path: str, timeout: int = 120):
    """
    Execute all cells and capture outputs (ATEFEH-Agent - Step 4a).
    """
    
    PIPELINE_FILENAME = "atefeh-pipeline.ipynb"
    if Path(notebook_path).name == PIPELINE_FILENAME:
        logger.error(
            "[ATEFEH] Refusing to execute the pipeline notebook on itself "
            "(self-reference). Point NOTEBOOK_PATH / NOTEBOOK_LIST at a "
            "target ML notebook instead."
        )
        raise ValueError("Self-reference detected: cannot execute the pipeline on itself.")

    logger.info("[ATEFEH] Starting notebook execution")

    ep = ExecutePreprocessor(
        timeout=timeout,
        kernel_name="python3"
    )

    notebook_dir = str(Path(notebook_path).parent)

    try:
        ep.preprocess(nb, {"metadata": {"path": notebook_dir}})
        logger.info("[ATEFEH] Notebook execution completed successfully")
    except Exception as e:
        logger.warning(f"[ATEFEH] Execution warning (partial outputs saved): {e}")
    except KeyboardInterrupt:
        logger.error("[ATEFEH] Execution interrupted by user")

    try:
        executed_path = OUTPUT_DIR / f"{Path(notebook_path).stem}_executed.ipynb"
        with open(executed_path, "w", encoding="utf-8") as f:
            nbformat.write(nb, f)
        logger.info(f"[ATEFEH] Executed notebook saved: {executed_path}")
    except Exception as save_error:
        logger.error(f"[ATEFEH] Failed to save executed notebook: {save_error}")

    return nb

2026-07-19 18:31:31,202 - INFO - ATEFEH batch started | total=27
2026-07-19 18:31:31,204 - INFO - [1/27] Processing: kaggle_notebooks/nyc_taxi_fare_prediction.ipynb | task: regression
2026-07-19 18:31:31,206 - INFO - ============================================================
2026-07-19 18:31:31,208 - INFO - ATEFEH pipeline started
2026-07-19 18:31:31,209 - INFO - ============================================================
2026-07-19 18:31:31,211 - INFO - [Step 1] User-Agent: Submit ML Program
2026-07-19 18:31:31,213 - INFO - [User-Agent] Submitting notebook: kaggle_notebooks/nyc_taxi_fare_prediction.ipynb
2026-07-19 18:31:31,215 - INFO - [User-Agent] Submission ready | title=nyc_taxi_fare_prediction | size=1983.7 KB
2026-07-19 18:31:31,216 - INFO - [Step 2] ATEFEH-Agent: Validate
2026-07-19 18:31:31,218 - INFO - [ATEFEH] Starting input validation
2026-07-19 18:31:31,256 - INFO - [ATEFEH] Input validation successful | code_cells=68
2026-07-19 18:31:31,259 - INFO - [Step 4a] ATEFEH-Ag

<h2 style="color:#2C3E50;">
Cell 18 — Load and Display All Results
</h2>

<p style="font-size:16px; line-height:1.9; text-align:justify;">

This cell reloads the combined results generated by the batch execution process (<code>run_all_notebooks</code>) and presents a structured summary for each processed notebook. It enables quick inspection of pipeline outcomes without re-running the entire system.

</p>

<h3 style="color:#34495E;">Purpose</h3>

<p style="font-size:16px; line-height:1.9;">

The goal of this step is to provide a human-readable overview of batch execution results, including success status, skipped files, and runtime errors.

</p>

<h3 style="color:#34495E;">Status Labels</h3>

<ul style="font-size:16px; line-height:1.9;">
<li><b>[OK]</b> — Successful execution; displays notebook name, task, number of extracted cells, and a 120-character preview of the explanation.</li>
<li><b>[SKIPPED]</b> — Notebook file was not found; execution was skipped.</li>
<li><b>[ERROR]</b> — Execution failed due to an unhandled exception during pipeline processing.</li>
</ul>

<div style="
background-color:#F8F9F9;
padding:14px;
border-left:5px solid #5DADE2;
border-radius:8px;
margin-top:10px;
font-size:15px;
line-height:1.8;">

<b>Note:</b><br>
This cell can be executed independently to inspect previously saved batch results without re-running the full ATEFEH pipeline, making it useful for debugging and analysis.

</div>

In [45]:
all_results_file = OUTPUT_DIR / "all_results.json"
 
# Load the combined results from disk
with open(all_results_file, "r", encoding="utf-8") as f:
    loaded_all = json.load(f)   # deserialize JSON list → Python list of dicts
 
logger.info(f"Loaded {len(loaded_all)} results from {all_results_file}")
logger.info("=" * 60)
 
# --- Loop over every result and print a formatted summary line ---
for r in loaded_all:
    # Extract fields safely with .get() defaults in case keys are missing
    status   = r.get("status",          "unknown")
    notebook = r.get("notebook",        "?")
    task     = r.get("task",            "?")
    cells    = r.get("cells_extracted", 0)
    message  = r.get("message",         "")
    expl     = r.get("explanation",     "")[:120]   # first 120 chars as a quick preview
 
    # --- Branch on status to choose the appropriate log level and format ---
    if status == "success":
        # Left-justify notebook name to 45 chars for aligned columns
        logger.info(
            f"[OK]      {notebook:<45} "
            f"| task={task:<18} | cells={cells}"
        )
        logger.info(f"          explanation: {expl}...")   # indented preview line
 
    elif status == "skipped":
        logger.warning(f"[SKIPPED] {notebook:<45} | {message}")
 
    else:
        # Covers "error" and any unexpected status values
        logger.error(f"[ERROR]   {notebook:<45} | {message}")
 
logger.info("=" * 60)

2026-07-19 20:02:16,006 - INFO - Loaded 27 results from notebook_outputs\all_results.json
2026-07-19 20:02:16,013 - INFO - ============================================================
2026-07-19 20:02:16,016 - INFO - [OK]      nyc_taxi_fare_prediction                      | task=regression         | cells=68
2026-07-19 20:02:16,018 - INFO -           explanation: **Task**

This notebook addresses the task of predicting taxi fares in New York City using machine learning models.

**R...
2026-07-19 20:02:16,024 - INFO - [OK]      tps_feb_2021_lgbmregressor                    | task=regression         | cells=15
2026-07-19 20:02:16,025 - INFO -           explanation: **Task**
This notebook addresses the task of regression analysis on a imbalanced dataset using LightGBM and LGBMRegresso...
2026-07-19 20:02:16,027 - INFO - [OK]      concrete_compressive_strength                 | task=regression         | cells=43
2026-07-19 20:02:16,028 - INFO -           explanation: **Task**
This notebook